# Modelo de apoio ao remanejamento de medicamentos
**Rafael Munarin · TCC · 07/10/2026**

Neste notebook, identifico lotes com risco de vencimento e unidades que podem receber parte desse estoque. Uso regras de consumo médio, validade e PVPS (Primeiro que Vence, Primeiro que Sai).

O modelo é determinístico: as mesmas entradas geram os mesmos resultados. Validade e saídas mensais são sintéticas; cadastro e saldo vêm das planilhas fornecidas.

**Para executar:** abra no Colab, execute as células em ordem e envie `CNES.xlsx` e `Consulta de Estoque.xlsx`. As saídas salvas são da execução local anterior; esta revisão de escrita mantém os cálculos.

| CRISP-DM | Etapas do notebook |
|---|---|
| Compreensão do negócio | Objetivo e regras: seção 1 |
| Compreensão dos dados | Fontes e recorte: seções 3–10 |
| Preparação dos dados | Consolidação e dados sintéticos: seções 11–14 |
| Modelagem | CMM, PVPS, risco e remanejamento: seções 15–23 |
| Avaliação | Cenários, invariantes, sensibilidade e reprodução: seção 24 |
| Apresentação | Consulta e exportação: seção 25; interface no Streamlit |


## 1. Objetivo e regras
O objetivo é identificar risco de vencimento e oportunidades de remanejamento. Uso o termo **diagnóstico** para identificar e quantificar essas situações, sem explicar suas causas.

1. Calculo o CMM dos seis meses completos anteriores à data de referência.
2. Aplico o PVPS por validade; em empate, uso o menor ID do lote.
3. Comparo outras unidades para o mesmo medicamento, com CMM positivo e capacidade de receber ao menos uma unidade inteira.
4. Preservo o estoque utilizável na origem e evito aumentar o risco no destino.
5. Escolho a maior quantidade admissível; em empate, a maior capacidade livre e, depois, o menor ID da unidade.
6. Atualizo os saldos após cada sugestão. O que foi recebido não é reenviado na mesma execução.

O cálculo considera toda a rede. Os filtros servem apenas para consulta. Uso a posição fixa de **09/06/2026**; uma nova execução semanal exigiria outra posição de estoque, e o CMM mudaria após um novo mês completo. Não há agendamento automático.

As quantidades recomendadas são inteiras, na unidade de fornecimento de cada produto. Frações de risco continuam visíveis. A ordem das escolhas pode afetar o plano; não busco a melhor solução global. A decisão de transferir permanece com o responsável.


## 2. Ambiente
### 2.1 Instalação
Instalo as bibliotecas de dados que estiverem faltando.


In [1]:
import importlib.util
import subprocess
import sys

pacotes = ['numpy', 'pandas', 'openpyxl']
faltantes = [p for p in pacotes if importlib.util.find_spec(p) is None]
if faltantes:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *faltantes])
else:
    print('Bibliotecas necessárias disponíveis.')


Bibliotecas necessárias disponíveis.


### 2.2 Importações
Carrego as bibliotecas usadas na leitura, nos cálculos e na exportação.


In [2]:
from pathlib import Path
import hashlib
import json
import re
import unicodedata
import zipfile
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 14)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

import uuid
if '_ID_SESSAO' not in globals():
    _ID_SESSAO = uuid.uuid4().hex


## 3. Fontes e parâmetros
| Informação | Origem |
|---|---|
| Unidades, produtos, lotes e saldo | Planilhas fornecidas |
| Validade e saídas mensais | Geração sintética |
| CMM, risco, capacidade e sugestão | Cálculos do modelo |

Não uso dados de pacientes nem trato a variação de saldo como consumo. As faixas de geração são escolhas do experimento, mantidas da versão de 01/10. Elas não foram estimadas nas UBS; avalio sua influência na seção 24.


In [3]:
PARAMETROS = {
    'referencia': '2026-06-09',
    'dias_mes': 30.0,
    'validade_min': 30,
    'validade_max': 540,
    'sal': 'TCC_ESTOQUE_20260609',
    'consumo_min': 5,
    'consumo_max': 120,
    'percentual_zero': 15,
}
DATA_REFERENCIA = pd.Timestamp(PARAMETROS['referencia'])
TOL = TOLERANCIA = 1e-8  # tolerância para arredondamento
CHAVE = ['id_unidade', 'id_medicamento']
CAMPOS_ESTOQUE = ['id_lote','id_unidade','id_medicamento','quantidade','validade']


In [4]:
# Interrompo a execução quando uma condição não é atendida.
def exigir(condicao, mensagem):
    if not bool(condicao):
        raise ValueError(mensagem)

def conferir_proximidade(obtido, esperado, mensagem):
    exigir(np.allclose(obtido, esperado, atol=TOLERANCIA, rtol=0, equal_nan=False), mensagem)


## 4. Entrada das planilhas
Uso as duas planilhas indicadas. Se houver cópias com nomes equivalentes, mantenho apenas a versão desejada.


In [5]:
# Carrego as planilhas que ainda não estão disponíveis.
PASTA_DADOS = Path(".")
PADROES_ARQUIVOS = [r"CNES(?:\s*\(\d+\))?\.xlsx", r"Consulta de Estoque(?:\s*\(\d+\))?\.xlsx"]

def localizar_planilhas():
    nomes = [p.name for p in PASTA_DADOS.iterdir() if p.is_file()]
    encontrados = []

    for padrao in PADROES_ARQUIVOS:
        candidatos = sorted(n for n in nomes if re.fullmatch(padrao, n, flags=re.IGNORECASE))
        exigir(len(candidatos) <= 1, f"Há arquivos equivalentes: {candidatos}. Mantenha apenas a fonte desejada.")
        encontrados.append(candidatos[0] if candidatos else None)
    return encontrados

ARQUIVOS = localizar_planilhas()

if any(nome is None for nome in ARQUIVOS):
    faltantes = [rotulo for rotulo, nome in zip(["CNES.xlsx", "Consulta de Estoque.xlsx"], ARQUIVOS) if nome is None]
    try:
        from google.colab import files
    except ImportError as erro:
        raise FileNotFoundError(f"Coloque os arquivos em PASTA_DADOS: {faltantes}") from erro
    print("Envie as planilhas que faltam:", ", ".join(faltantes))
    enviados = files.upload()

    # O upload do Colab grava os arquivos no diretório de trabalho.
    del enviados
    ARQUIVOS = localizar_planilhas()
    exigir(all(nome is not None for nome in ARQUIVOS), "Ainda falta uma das planilhas necessárias.")


## 5. Leitura
No arquivo de estoque, o cabeçalho começa na quarta linha.


In [6]:
# Leio as duas fontes.
df_cnes = pd.read_excel(PASTA_DADOS / ARQUIVOS[0])
df_estoque_bruto = pd.read_excel(PASTA_DADOS / ARQUIVOS[1], header=3)
print('Linhas lidas:', len(df_cnes), 'no CNES e', len(df_estoque_bruto), 'no relatório de estoque.')


Linhas lidas: 518 no CNES e 14122 no relatório de estoque.


## 6. Limpeza dos dados
Retiro linhas vazias e rodapés conhecidos. Dados inesperados interrompem a execução. Mantenho a linha original para rastrear cada registro.


In [7]:
# Separo linhas vazias e rodapés conhecidos.
def separar_linhas_estoque(bruto):
    exigir("Produto - Código" in bruto.columns, "Código de produto ausente do cabeçalho.")
    valores = pd.to_numeric(bruto["Produto - Código"], errors="coerce")
    rotulos = bruto["Produto - Código"].fillna("").astype(str).str.strip()
    conhecidos = rotulos.str.fullmatch(r"Total de Registros:|IPM Sistemas Ltda|Atende\.Net - EST v:[0-9.]+")
    campos = [c for c in bruto if not str(c).startswith("Unnamed:") and c != "Produto - Código"]
    rodape = valores.isna() & conhecidos & bruto[campos].isna().all(axis=1)
    vazias = bruto.isna().all(axis=1)
    desconhecidas = valores.isna() & ~(rodape | vazias)
    exigir(not desconhecidas.any(), f"Linhas com produto inválido exigem revisão: {(bruto.index[desconhecidas]+5).tolist()[:20]}")
    removidas = bruto.loc[rodape | vazias].copy()
    removidas["linha_excel_origem"] = removidas.index + 5
    removidas["motivo_exclusao"] = np.where(vazias.loc[removidas.index], "LINHA_VAZIA", "RODAPE_RECONHECIDO")
    return bruto.loc[~(rodape | vazias)].copy(), removidas


In [8]:
# Confiro as colunas e retiro os rodapés.
COLUNAS_ESTOQUE = ["Produto - Código", "Produto - Descrição", "Lote", "Código", "Lote - Código",
                  "Quantidade", "Estabelecimento - Nome", "Estabelecimento - Código",
                  "Grupo de Produto - Código", "Grupo de Produto - Descrição"]

exigir(set(COLUNAS_ESTOQUE).issubset(df_estoque_bruto.columns), "Cabeçalho do estoque diferente do esperado.")
df_estoque, df_linhas_excluidas = separar_linhas_estoque(df_estoque_bruto)
administrativas_removidas = len(df_linhas_excluidas)

# Confiro as colunas sem nome antes de removê-las.
unnamed = [c for c in df_estoque.columns if str(c).startswith("Unnamed:")]
exigir(df_estoque[unnamed].isna().all().all(), "Há conteúdo em colunas Unnamed: revisar o relatório.")
df_estoque = df_estoque[COLUNAS_ESTOQUE].copy()

# Guardo a linha de origem e renomeio as colunas.
df_estoque["linha_excel_origem"] = df_estoque.index + 5
df_estoque = df_estoque.rename(columns={
    "Produto - Código": "codigo_produto", "Produto - Descrição": "descricao_produto",
    "Lote": "lote", "Código": "codigo_registro_estoque", "Lote - Código": "codigo_lote",
    "Quantidade": "quantidade_estoque", "Estabelecimento - Nome": "nome_estabelecimento_portal",
    "Estabelecimento - Código": "codigo_estabelecimento",
    "Grupo de Produto - Código": "grupo_produto_codigo", "Grupo de Produto - Descrição": "grupo_produto_descricao"
})


In [9]:
# Converto os códigos e confiro se são válidos.
for coluna in ["codigo_produto", "codigo_registro_estoque", "codigo_lote", "codigo_estabelecimento", "grupo_produto_codigo"]:
    valores = pd.to_numeric(df_estoque[coluna], errors="raise")
    exigir(valores.notna().all() and np.isfinite(valores).all() and (valores % 1 == 0).all(), f"Código inválido: {coluna}")
    df_estoque[coluna] = valores.astype("int64")

df_estoque["quantidade_estoque"] = pd.to_numeric(df_estoque["quantidade_estoque"], errors="raise")


## 7. Padronização dos nomes
Padronizo acentos, letras e espaços para relacionar as duas fontes.


In [10]:
# Padronizo os nomes das unidades.
def normalizar_texto(valor):
    if pd.isna(valor):
        return ""
    texto = unicodedata.normalize("NFKD", str(valor).strip().upper())
    return re.sub(r"\s+", " ", "".join(c for c in texto if not unicodedata.combining(c)))


## 8. Recorte da base
Mantenho as 17 unidades e os grupos definidos no modelo. Nesta fonte, o grupo 41 contém adesivos e goma de nicotina e bupropiona. Se a fonte mudar, esse recorte precisa ser conferido.


In [11]:
# Defino o recorte de unidades e produtos.
UNIDADES_SELECIONADAS = [
    "EAP ITOUPAVA", "ESF SILVIO SCHUTZ BAIRRO SUMARE", "PSF BARRA DO TROMBUDO",
    "PSF BARRAGEM CAIC", "PSF BELA ALIANCA", "PSF BOA VISTAEUGENIO SCHNEIDER",
    "PSF BREMER", "PSF BUDAG", "PSF CANOAS", "PSF CANTA GALO", "PSF FUNDO CANOAS",
    "PSF LARANJEIRAS", "PSF PAMPLONA", "PSF PROGRESSO", "PSF SANTA RITA", "PSF SANTANA", "PSF TABOAO"
]
GRUPOS_MEDICAMENTOS = {6: "Medicamentos", 24: "MED PAI", 26: "MED ANTIBIOTICOS",
                      27: "MED BASICOS", 41: "TABAGISMO medicamentos e Insumos"}

ordem = {normalizar_texto(nome): i for i, nome in enumerate(UNIDADES_SELECIONADAS, 1)}
exigir(len(ordem) == len(UNIDADES_SELECIONADAS), "Unidades selecionadas repetidas.")


## 9. Integração do CNES com o estoque
Relaciono as unidades entre as fontes e mantenho o CNES, o código original e um ID interno.


In [12]:
# Padronizo os nomes nas duas fontes.
exigir({"NOME FANTASIA", "CNES"}.issubset(df_cnes.columns), "Cabeçalho CNES inválido.")
cnes_normalizado = df_cnes.assign(nome_normalizado=df_cnes["NOME FANTASIA"].map(normalizar_texto))
cnes_sel = cnes_normalizado.loc[cnes_normalizado.nome_normalizado.isin(ordem), ["CNES", "NOME FANTASIA", "nome_normalizado"]].copy()
portal = df_estoque[["codigo_estabelecimento", "nome_estabelecimento_portal"]].drop_duplicates().copy()
portal["nome_normalizado"] = portal.nome_estabelecimento_portal.map(normalizar_texto)
portal = portal.loc[portal.nome_normalizado.isin(ordem)]

# Relaciono as fontes e confiro os identificadores.
df_cadastro_unidades = cnes_sel.merge(portal, on="nome_normalizado", how="left", validate="one_to_one")
exigir(set(df_cadastro_unidades.nome_normalizado) == set(ordem), "Uma ou mais unidades não foram localizadas no CNES.")
exigir(df_cadastro_unidades.codigo_estabelecimento.notna().all(), "Unidade sem correspondência no estoque.")
df_cadastro_unidades = df_cadastro_unidades.rename(columns={"CNES": "cnes", "NOME FANTASIA": "nome_unidade"})
df_cadastro_unidades["id_unidade"] = df_cadastro_unidades.nome_normalizado.map(ordem).astype(int)

for c in ["cnes", "codigo_estabelecimento"]:
    valores = pd.to_numeric(df_cadastro_unidades[c], errors="raise")
    exigir(valores.notna().all() and (valores % 1 == 0).all(), f"Identificador inválido: {c}")
    df_cadastro_unidades[c] = valores.astype("int64")
    exigir(df_cadastro_unidades[c].is_unique, f"Identificador duplicado: {c}")

df_cadastro_unidades = df_cadastro_unidades[[
    "id_unidade", "cnes", "codigo_estabelecimento", "nome_unidade", "nome_estabelecimento_portal"
    ]].sort_values("id_unidade").reset_index(drop=True)


## 10. Catálogo e registros selecionados
Confiro códigos, lotes e saldos. Registros repetidos precisam ser revisados antes de qualquer soma.


In [13]:
# Seleciono os registros e confiro seus dados.
selecionados = df_estoque.loc[
    df_estoque.codigo_estabelecimento.isin(df_cadastro_unidades.codigo_estabelecimento)
    & df_estoque.grupo_produto_codigo.isin(GRUPOS_MEDICAMENTOS)
].copy()

exigir(len(selecionados) > 0, "Seleção sem registros.")
exigir(selecionados[["lote", "descricao_produto", "grupo_produto_descricao"]].notna().all().all(), "Campos cadastrais ausentes.")
selecionados["lote"] = selecionados.lote.astype(str)
exigir(selecionados.lote.str.strip().ne("").all(), "Lote vazio.")
exigir(np.isfinite(selecionados.quantidade_estoque).all() and selecionados.quantidade_estoque.gt(0).all(), "Estoque deve ser finito e positivo nesta extração.")
exigir(selecionados.codigo_registro_estoque.is_unique, "Código de registro repetido: revisar antes de somar.")

# Monto o catálogo de medicamentos.
df_medicamentos = selecionados[[
    "codigo_produto", "descricao_produto", "grupo_produto_codigo", "grupo_produto_descricao"
    ]].drop_duplicates().sort_values("codigo_produto").reset_index(drop=True)

exigir(df_medicamentos.codigo_produto.is_unique, "Produto com atributos cadastrais divergentes.")
df_medicamentos.insert(0, "id_medicamento", np.arange(1, len(df_medicamentos) + 1))

# Incluo os IDs de unidade e medicamento no estoque.
df_registros_estoque = selecionados.merge(
    df_cadastro_unidades[["codigo_estabelecimento", "id_unidade"]], on="codigo_estabelecimento", validate="many_to_one"
).merge(df_medicamentos[["codigo_produto", "id_medicamento"]], on="codigo_produto", validate="many_to_one")
print('Unidades:', len(df_cadastro_unidades), '| Produtos:', len(df_medicamentos), '| Registros selecionados:', len(df_registros_estoque))


Unidades: 17 | Produtos: 161 | Registros selecionados: 4174


## 11. Consolidação
Agrupo registros da mesma unidade, medicamento, lote e código de lote. Depois, confiro se as quantidades foram preservadas.


In [14]:
# Agrupo os registros de cada lote.
CHAVE_ANALITICA = ["id_unidade", "id_medicamento", "lote", "codigo_lote"]

df_estoque_consolidado = df_registros_estoque.groupby(CHAVE_ANALITICA, as_index=False, dropna=False).agg(
    quantidade_estoque=("quantidade_estoque", "sum"),
    quantidade_registros_origem=("codigo_registro_estoque", "size")
).sort_values(CHAVE_ANALITICA).reset_index(drop=True)

df_estoque_consolidado.insert(0, "id_estoque_lote", np.arange(1, len(df_estoque_consolidado) + 1))
df_estoque_consolidado = df_estoque_consolidado.merge(df_medicamentos, on="id_medicamento", validate="many_to_one").merge(
    df_cadastro_unidades[["id_unidade", "cnes", "nome_unidade"]], on="id_unidade", validate="many_to_one"
    )
df_estoque_consolidado["data_referencia_estoque"] = DATA_REFERENCIA
df_estoque_consolidado["natureza_estoque"] = "PUBLICO_DERIVADO_POR_AGREGACAO"


In [15]:
# Mantenho a ligação com as linhas da fonte.
df_rastreabilidade = df_registros_estoque[CHAVE_ANALITICA + ["codigo_registro_estoque", "linha_excel_origem", "quantidade_estoque"]].merge(
    df_estoque_consolidado[CHAVE_ANALITICA + ["id_estoque_lote"]], on=CHAVE_ANALITICA, validate="many_to_one"
    )

# Confiro se o agrupamento preservou as quantidades.
exigir(not df_estoque_consolidado.duplicated(CHAVE_ANALITICA).any(), "Chave analítica duplicada após consolidação.")
conferir_proximidade(df_estoque_consolidado.quantidade_estoque.sum(), df_registros_estoque.quantidade_estoque.sum(), "Soma de estoque alterada.")

controle_antes = df_registros_estoque.groupby(["id_unidade", "id_medicamento"]).quantidade_estoque.sum().sort_index()
controle_depois = df_estoque_consolidado.groupby(["id_unidade", "id_medicamento"]).quantidade_estoque.sum().sort_index()

exigir(controle_antes.index.equals(controle_depois.index), "Combinações alteradas na consolidação.")
conferir_proximidade(controle_antes, controle_depois, "Estoque alterado por unidade–medicamento.")


## 12. Geração reproduzível com SHA-256
Uso o hash para transformar uma chave em um número fixo. Depois, aplico a faixa definida nos parâmetros.

- **Validade:** sal, produto, lote e código do lote. O mesmo lote tem a mesma validade na rede.
- **Consumo:** sal, CNES e produto. No histórico, acrescento o índice do mês.

Mantenho as chaves da versão de 01/10. O hash permite repetir os dados; não comprova que eles representam a realidade e não participa da escolha dos destinos.


In [16]:
# Gero um número fixo para cada chave.
def hash_inteiro(*partes):
    # Uso JSON para separar os campos da chave.
    chave = json.dumps([str(p) for p in partes], ensure_ascii=False, separators=(",", ":"))
    return int(hashlib.sha256(chave.encode("utf-8")).hexdigest()[:16], 16)


In [17]:
# Confiro as faixas do gerador.
def validar_consumo_simulado(minimo, maximo, percentual_zero):
    exigir(all(isinstance(v, (int, np.integer)) and not isinstance(v, (bool, np.bool_))
               for v in [minimo, maximo, percentual_zero]), "Parâmetros de consumo devem ser inteiros.")
    exigir(0 < minimo <= maximo, "Faixa de consumo deve ser positiva e ordenada.")
    exigir(0 <= percentual_zero <= 100, "Percentual de consumo zero fora de 0–100.")


## 13. Combinações e consumo-base
Monto todas as combinações entre unidades e medicamentos, inclusive as que não têm estoque. O parâmetro de 15% atua no consumo-base; o percentual final de CMM zero pode ser diferente.


In [18]:
# Incluo todas as combinações, mesmo sem estoque.
grade = df_cadastro_unidades[["id_unidade", "cnes"]].merge(
    df_medicamentos[["id_medicamento", "codigo_produto"]], how="cross"
    )


In [19]:
# Gero o consumo-base com os parâmetros definidos.
def gerar_consumo_base(grade, semente, minimo=5, maximo=120, percentual_zero=15):
    validar_consumo_simulado(minimo, maximo, percentual_zero)
    hashes = [hash_inteiro(semente, "consumo", c, p)
              for c, p in grade[["cnes", "codigo_produto"]].itertuples(index=False, name=None)]
    return np.array([0 if h % 100 < percentual_zero else minimo + (h // 100) % (maximo-minimo+1)
                     for h in hashes], dtype="int64")


## 14. Validade e histórico sintéticos
Gero seis meses completos: dezembro/2025 a maio/2026. Mantenho os perfis estável, crescente, decrescente e intermitente. O rótulo `RUPTURA` equivale ao estável nessa janela.

Os fatores do gerador são: variação de 0,75 a 1,25; tendência de 0,015; piso de 0,15; intermitência de 60% com fator 2,5. São escolhas sintéticas, sem validação nas UBS. Não gero meses futuros nem recalculo o saldo público a partir dessas saídas. O gerador não consulta os resultados do modelo.


In [20]:
def gerar_series_temporais(grade_base, semente, n_historico=6):
    """Gero o histórico sem consultar saldos ou resultados."""
    perfis = ['ESTAVEL', 'CRESCENTE', 'DECRESCENTE', 'INTERMITENTE', 'RUPTURA']
    registros = []
    for r in grade_base.sort_values(['id_unidade', 'id_medicamento']).itertuples(index=False):
        perfil = perfis[hash_inteiro(semente, 'perfil', r.cnes, r.codigo_produto) % len(perfis)]
        if r.quantidade_base_simulada == 0:
            perfil = 'ZERO'
        for t in range(-n_historico, 0):
            h = hash_inteiro(semente, 'ruido', r.cnes, r.codigo_produto, t)
            ruido = 0.75 + (h % 10001) / 10000 * 0.5
            fator = 1.0
            if perfil == 'CRESCENTE':
                fator = max(0.15, 1.0 + 0.015 * t)
            elif perfil == 'DECRESCENTE':
                fator = max(0.15, 1.0 - 0.015 * t)
            elif perfil == 'INTERMITENTE':
                fator = 0.0 if (h // 10001) % 100 < 60 else 2.5
            elif perfil == 'RUPTURA':
                # Nesta janela, RUPTURA usa o mesmo fator do perfil estável.
                fator = 1.0
            quantidade = int(np.rint(r.quantidade_base_simulada * fator * ruido))
            registros.append((int(r.id_unidade), int(r.id_medicamento), t, perfil, quantidade))
    return pd.DataFrame(registros, columns=['id_unidade', 'id_medicamento', 'indice_tempo',
                                          'perfil_simulado', 'quantidade_saida'])


In [21]:
def gerar_dados_experimentais(base, grade_original, parametros):
    """Gero validade e saídas com as mesmas chaves da base revisada."""
    p = parametros
    exigir(0 <= p['validade_min'] <= p['validade_max'], 'Faixa de validade inválida.')
    exigir(p['dias_mes'] > 0 and np.isfinite(p['dias_mes']), 'Dias por mês inválidos.')
    validar_consumo_simulado(p['consumo_min'], p['consumo_max'], p['percentual_zero'])
    exigir(isinstance(p['sal'], str) and bool(p['sal']), 'Sal do hash vazio.')
    estoque = base.copy()
    amplitude = p['validade_max'] - p['validade_min'] + 1
    dias = [p['validade_min'] + hash_inteiro(p['sal'], 'validade', produto, lote, codigo) % amplitude
            for produto, lote, codigo in estoque[['codigo_produto','lote','codigo_lote']].itertuples(index=False, name=None)]
    estoque['validade'] = pd.Timestamp(p['referencia']) + pd.to_timedelta(dias, unit='D')
    estoque = estoque.rename(columns={'id_estoque_lote':'id_lote', 'quantidade_estoque':'quantidade'})
    grade_local = grade_original.copy()
    grade_local['quantidade_base_simulada'] = gerar_consumo_base(grade_local, p['sal'],
        p['consumo_min'], p['consumo_max'], p['percentual_zero'])
    historico = gerar_series_temporais(grade_local, p['sal'], n_historico=6)
    periodos = pd.period_range(end=pd.Timestamp(p['referencia']).to_period('M')-1, periods=6)
    historico['periodo'] = historico.indice_tempo.map(dict(zip(range(-6,0), periodos.astype(str))))
    return estoque, historico, periodos


In [22]:
estoque, historico, periodos = gerar_dados_experimentais(df_estoque_consolidado, grade, PARAMETROS)
exigir(estoque.groupby(['codigo_produto','lote','codigo_lote']).validade.nunique().eq(1).all(),
       'Mesmo lote com validades diferentes na rede.')
print('Lotes consolidados:',len(estoque),'| Saídas mensais sintéticas:',len(historico))
display(estoque[['id_lote','nome_unidade','descricao_produto','lote','quantidade','validade']].head(5))


Lotes consolidados: 3846 | Saídas mensais sintéticas: 16422


,id_lote,nome_unidade,descricao_produto,lote,quantidade,validade
0,1,EAP ITOUPAVA,Cloridrato de lidocaina 2% geleia 30gr. (BISNAGA),252889,23.0000,2026-11-05
1,2,EAP ITOUPAVA,Cloridrato de lidocaina 2% geleia 30gr. (BISNAGA),253618,10.0000,2026-08-20
2,3,EAP ITOUPAVA,Cloridrato de lidocaina 2% geleia 30gr. (BISNAGA),50036035,9.0000,2027-07-29
3,4,EAP ITOUPAVA,"CARVEDILOL 12,5mg (COMPRIMIDO)",4S8705,1030.0000,2027-09-23
4,5,EAP ITOUPAVA,"CARVEDILOL 12,5mg (COMPRIMIDO)",4V1755,980.0000,2027-01-28


## 15. Consumo Médio Mensal
Uso a média das saídas dos seis meses:

$$CMM = (s_1+s_2+s_3+s_4+s_5+s_6)/6$$

Essa média é a taxa de referência do cálculo. Meses ausentes, repetidos ou valores negativos são tratados como erro; ausência não vira zero.


In [23]:
def calcular_cmm(historico, periodos):
    """Calculo a média dos seis meses completos."""
    campos = CHAVE + ['periodo', 'quantidade_saida']
    exigir(set(campos).issubset(historico), 'Histórico sem os campos necessários.')
    h = historico[campos].copy()
    h['periodo'] = h.periodo.astype(str)
    meses = [str(p) for p in periodos]
    exigir(len(meses) == 6 and len(set(meses)) == 6, 'São necessários seis meses distintos.')
    exigir(h[CHAVE].notna().all().all(), 'Histórico com identificação ausente.')
    exigir(not h.duplicated(CHAVE + ['periodo']).any(), 'Mês duplicado no histórico.')
    exigir(set(h.periodo) == set(meses), 'Histórico com mês ausente ou fora da janela.')
    h['quantidade_saida'] = pd.to_numeric(h.quantidade_saida, errors='raise')
    exigir(np.isfinite(h.quantidade_saida).all() and h.quantidade_saida.ge(0).all(),
           'Saída mensal deve ser finita e não negativa.')
    exigir(h.groupby(CHAVE).periodo.nunique().eq(6).all(), 'Combinação sem os seis meses.')
    return h.groupby(CHAVE, as_index=False).agg(
        cmm=('quantidade_saida', 'mean'), total_seis_meses=('quantidade_saida', 'sum'))


In [24]:
medias = calcular_cmm(historico, periodos)
display(medias.head(5))


,id_unidade,id_medicamento,cmm,total_seis_meses
0,1,1,5.8333,35
1,1,2,49.6667,298
2,1,3,4.3333,26
3,1,4,37.3333,224
4,1,5,0.0000,0


## 16. Conferência das entradas
Verifico se há saldo, validade e CMM para cada combinação. CMM zero é válido; CMM ausente interrompe o cálculo.


In [25]:
def validar_entradas(estoque, medias, referencia, dias_mes):
    exigir(set(CAMPOS_ESTOQUE).issubset(estoque), 'Estoque incompleto.')
    exigir(set(CHAVE + ['cmm']).issubset(medias), 'Tabela de CMM incompleta.')
    exigir(np.isfinite(dias_mes) and dias_mes > 0, 'Dias por mês inválidos.')
    exigir(not pd.isna(referencia), 'Data de referência ausente.')
    exigir(not medias.duplicated(CHAVE).any(), 'CMM duplicado.')
    exigir(medias[CHAVE].notna().all().all(), 'Identificador de CMM ausente.')
    exigir(np.isfinite(medias.cmm).all() and medias.cmm.ge(0).all(), 'CMM inválido.')
    exigir(estoque[['id_lote'] + CHAVE].notna().all().all(), 'Identificador de estoque ausente.')
    exigir(estoque.id_lote.is_unique, 'ID de lote duplicado.')
    exigir(np.isfinite(estoque.quantidade.to_numpy(dtype=float)).all() and estoque.quantidade.ge(0).all(), 'Saldo inválido.')
    exigir(pd.to_datetime(estoque.validade, errors='raise').notna().all(), 'Validade ausente.')
    pares = set(map(tuple, medias[CHAVE].to_numpy()))
    exigir(set(map(tuple, estoque[CHAVE].to_numpy())).issubset(pares),
           'Estoque sem CMM. Um dado ausente não pode ser convertido em zero.')


In [26]:
def organizar_grupos(estoque, medias):
    grupos = {tuple(chave): g.to_dict('records') for chave, g in estoque.groupby(CHAVE)}
    taxas = {(int(r.id_unidade), int(r.id_medicamento)): float(r.cmm)
             for r in medias.itertuples(index=False)}
    return grupos, taxas


## 17. Risco por lote com PVPS
Ordeno os lotes por validade e ID. Para cada prazo, calculo a capacidade acumulada:

$$C_i = CMM \times \max(0,d_i)/D$$

Aqui, $D$ representa os dias por mês. Desconto o consumo já reservado aos lotes anteriores:

$$u_i = \min(q_i,\max(0,C_i-\sum_{j<i}u_j)),\qquad r_i=q_i-u_i$$

Assim, não uso a mesma capacidade duas vezes. O risco é uma quantidade potencial, não uma probabilidade. Sem prazo útil, a capacidade é zero.


In [27]:
def alocar_pvps(lotes, cmm, referencia, dias_mes=30.0):
    """Distribuo a capacidade pela ordem de vencimento."""
    acumulado = 0.0
    resultado = []
    for ordem, lote in enumerate(sorted(lotes, key=lambda r: (r['validade'], r['id_lote'])), 1):
        dias = (pd.Timestamp(lote['validade']) - pd.Timestamp(referencia)).days
        capacidade = cmm * max(0, dias) / dias_mes
        consumivel = min(lote['quantidade'], max(0.0, capacidade - acumulado))
        resultado.append(dict(lote, ordem_pvps=ordem, dias_ate_validade=dias,
            capacidade_acumulada=capacidade, consumo_anterior=acumulado,
            consumivel=consumivel, risco=lote['quantidade']-consumivel))
        acumulado += consumivel
    return resultado


In [28]:
def diagnosticar_lotes(estoque, medias, referencia, dias_mes=30.0):
    validar_entradas(estoque, medias, referencia, dias_mes)
    grupos, taxas = organizar_grupos(estoque, medias)
    linhas = []
    for chave, lotes in sorted(grupos.items()):
        linhas.extend(alocar_pvps(lotes, taxas[chave], referencia, dias_mes))
    campos = CAMPOS_ESTOQUE + ['ordem_pvps', 'dias_ate_validade',
        'capacidade_acumulada', 'consumo_anterior', 'consumivel', 'risco']
    return pd.DataFrame(linhas, columns=campos).astype({c: float for c in
        ['quantidade', 'capacidade_acumulada', 'consumo_anterior', 'consumivel', 'risco']})


## 18. Estoque e cobertura
Calculo a cobertura em dias como `saldo / CMM × dias por mês`. Com CMM zero, a cobertura fica sem valor. Combinações sem estoque e com consumo também aparecem, como no cenário C7.


In [29]:
def diagnosticar_combinacoes(estoque, medias, dias_mes=30.0):
    # Parto do histórico para manter as combinações sem estoque.
    saldos = estoque.groupby(CHAVE, as_index=False).quantidade.sum()
    tabela = medias.merge(saldos, on=CHAVE, how='left', validate='one_to_one')
    tabela['quantidade'] = tabela.quantidade.astype(float).fillna(0.0)
    tabela['cobertura_dias'] = np.nan
    positivo = tabela.cmm.gt(0)
    tabela.loc[positivo, 'cobertura_dias'] = (
        tabela.loc[positivo, 'quantidade'] / tabela.loc[positivo, 'cmm'] * dias_mes)
    tabela['situacao'] = np.select([
        tabela.quantidade.eq(0) & positivo,
        tabela.quantidade.gt(0) & ~positivo,
        tabela.quantidade.eq(0) & ~positivo], [
        'INDISPONIVEL_COM_CONSUMO', 'ESTOQUE_SEM_CONSUMO', 'SEM_ESTOQUE_SEM_CONSUMO'],
        default='COM_ESTOQUE_E_CONSUMO')
    return tabela


## 19. Capacidade do destino
Calculo quanto a unidade pode consumir até a validade recebida e desconto o uso dos lotes próprios que vencem antes ou na mesma data. Essa é a **capacidade livre**.

Também confiro os lotes que vencem depois. O limite final preserva o consumo já reservado a eles: é a **capacidade sem novo risco**.

Exemplo: com CMM de 30 e 60 unidades vencendo em 60 dias, receber 30 unidades para vencer em 30 dias ocuparia o consumo necessário ao lote próprio. Nesse caso, a quantidade admissível é zero.


In [30]:
def capacidade_destino(proprios, cmm, validade_recebida, referencia, dias_mes=30.0):
    """Calculo quanto o destino pode receber sem aumentar seu risco."""
    prazo = max(0, (pd.Timestamp(validade_recebida) - pd.Timestamp(referencia)).days)
    diagnostico = alocar_pvps(proprios, cmm, referencia, dias_mes)
    prioritario = sum(r['consumivel'] for r in diagnostico if r['validade'] <= validade_recebida)
    livre = max(0.0, cmm * prazo / dias_mes - prioritario)
    # Preservo também o consumo dos lotes que vencem depois.
    folgas = [livre]
    for r in diagnostico:
        if r['validade'] > validade_recebida:
            folgas.append(max(0.0, r['capacidade_acumulada'] - r['consumo_anterior'] - r['consumivel']))
    return livre, min(folgas)


## 20. Unidades candidatas
A quantidade admissível é o menor valor entre risco e capacidade sem novo risco, arredondado para baixo. Registro os destinos descartados e seus motivos. A origem nunca é candidata a receber o próprio lote.


In [31]:
CAMPOS_CANDIDATAS = ['id_lote_origem', 'id_unidade_origem', 'id_medicamento', 'id_unidade_destino', 'cmm_destino', 'capacidade_livre', 'capacidade_sem_novo_risco', 'quantidade_admissivel', 'motivo']


In [32]:
def avaliar_candidatas(lote, risco, grupos, taxas, referencia, dias_mes=30.0):
    linhas = []
    prazo = (pd.Timestamp(lote['validade']) - pd.Timestamp(referencia)).days
    for (unidade, medicamento), cmm in sorted(taxas.items()):
        if medicamento != lote['id_medicamento'] or unidade == lote['id_unidade']:
            continue
        livre, segura = capacidade_destino(grupos.get((unidade, medicamento), []),
            cmm, lote['validade'], referencia, dias_mes)
        quantidade = max(0, int(np.floor(min(risco, segura) + TOL))) if prazo > 0 else 0
        if prazo <= 0:
            motivo = 'SEM_PRAZO_UTIL'
        elif cmm == 0:
            motivo = 'CMM_ZERO'
        elif quantidade == 0:
            motivo = 'CAPACIDADE_OU_RISCO_INFERIOR_A_UMA_UNIDADE'
        else:
            motivo = 'ADMISSIVEL'
        linhas.append({'id_lote_origem': lote['id_lote'],
            'id_unidade_origem': lote['id_unidade'], 'id_medicamento': medicamento,
            'id_unidade_destino': unidade, 'cmm_destino': cmm,
            'capacidade_livre': livre, 'capacidade_sem_novo_risco': segura,
            'quantidade_admissivel': quantidade, 'motivo': motivo})
    return sorted(linhas, key=lambda r: (-r['quantidade_admissivel'], -r['capacidade_livre'],
                                       r['id_unidade_destino']))


In [33]:
def candidatas_iniciais(estoque, medias, diagnostico, referencia, dias_mes=30.0):
    grupos, taxas = organizar_grupos(estoque, medias)
    linhas = []
    for lote in diagnostico.loc[diagnostico.risco.gt(TOL)].to_dict('records'):
        linhas.extend(avaliar_candidatas(lote, lote['risco'], grupos, taxas, referencia, dias_mes))
    return pd.DataFrame(linhas, columns=CAMPOS_CANDIDATAS)


## 21. Plano de remanejamento
Começo pela menor validade; em empate, pelo menor ID de medicamento e lote. Atualizo os saldos após cada sugestão e recalculo a capacidade disponível. Um lote pode atender mais de um destino.

As candidatas iniciais são possibilidades isoladas. Não somo essas capacidades como se todas as transferências pudessem ocorrer juntas. O plano considera a disputa pela capacidade e todos os prazos positivos. Os movimentos representam sugestões, não transferências realizadas.


In [34]:
def construir_plano(estoque, medias, referencia, dias_mes=30.0):
    """Atualizo os saldos após cada sugestão."""
    validar_entradas(estoque, medias, referencia, dias_mes)
    grupos, taxas = organizar_grupos(estoque[CAMPOS_ESTOQUE], medias)
    estado = [r for lotes in grupos.values() for r in lotes]
    originais = sorted(estado, key=lambda r: (r['validade'], r['id_medicamento'], r['id_lote']))
    novo_id = int(estoque.id_lote.max()) + 1 if len(estoque) else 1
    plano, alternativas = [], []
    etapa = 0
    for lote in originais:
        origem = (lote['id_unidade'], lote['id_medicamento'])
        while True:
            diag_origem = alocar_pvps(grupos[origem], taxas[origem], referencia, dias_mes)
            risco = next(r['risco'] for r in diag_origem if r['id_lote'] == lote['id_lote'])
            if risco + TOL < 1 or lote['validade'] <= referencia:
                break
            candidatas = avaliar_candidatas(lote, risco, grupos, taxas, referencia, dias_mes)
            etapa += 1
            alternativas.extend(dict(r, etapa=etapa) for r in candidatas)
            admissiveis = [r for r in candidatas if r['quantidade_admissivel'] > 0]
            if not admissiveis:
                break
            escolhida = admissiveis[0]
            destino = (escolhida['id_unidade_destino'], lote['id_medicamento'])
            quantidade = escolhida['quantidade_admissivel']
            consumo_antes = sum(r['consumivel'] for r in diag_origem)
            destino_antes = alocar_pvps(grupos.get(destino, []), taxas[destino], referencia, dias_mes)
            recebido = dict(lote, id_lote=novo_id, id_unidade=destino[0], quantidade=float(quantidade))
            lote['quantidade'] -= quantidade
            grupos.setdefault(destino, []).append(recebido)
            estado.append(recebido)
            origem_depois = alocar_pvps(grupos[origem], taxas[origem], referencia, dias_mes)
            destino_depois = alocar_pvps(grupos[destino], taxas[destino], referencia, dias_mes)
            plano.append(dict(escolhida, etapa=etapa, ordem_movimento=len(plano)+1,
                id_lote_recebido=novo_id, validade=lote['validade'], risco_antes=risco,
                consumo_origem_antes=consumo_antes,
                consumo_origem_depois=sum(r['consumivel'] for r in origem_depois),
                risco_destino_antes=sum(r['risco'] for r in destino_antes),
                risco_destino_depois=sum(r['risco'] for r in destino_depois)))
            novo_id += 1
    campos = CAMPOS_CANDIDATAS + ['etapa', 'ordem_movimento', 'id_lote_recebido',
        'validade', 'risco_antes', 'consumo_origem_antes', 'consumo_origem_depois',
        'risco_destino_antes', 'risco_destino_depois']
    return (pd.DataFrame(plano, columns=campos), pd.DataFrame(estado, columns=CAMPOS_ESTOQUE),
            pd.DataFrame(alternativas, columns=['etapa']+CAMPOS_CANDIDATAS))


## 22. Execução do modelo
Executo o diagnóstico e monto o plano para toda a base. A função de execução também será usada na análise de sensibilidade.


In [35]:
diagnostico = diagnosticar_lotes(estoque, medias, DATA_REFERENCIA, PARAMETROS['dias_mes'])
combinacoes = diagnosticar_combinacoes(estoque, medias, PARAMETROS['dias_mes'])
candidatas = candidatas_iniciais(estoque, medias, diagnostico, DATA_REFERENCIA, PARAMETROS['dias_mes'])
plano, estoque_final, alternativas = construir_plano(
    estoque, medias, DATA_REFERENCIA, PARAMETROS['dias_mes'])
resultado = dict(estoque=estoque, historico=historico, medias=medias, diagnostico=diagnostico,
    combinacoes=combinacoes, candidatas=candidatas, plano=plano, final=estoque_final,
    alternativas=alternativas)


In [36]:
def executar_cenario_base(base, grade_original, parametros):
    estoque, historico, periodos = gerar_dados_experimentais(base, grade_original, parametros)
    medias = calcular_cmm(historico, periodos)
    ref, dias = pd.Timestamp(parametros['referencia']), parametros['dias_mes']
    diagnostico = diagnosticar_lotes(estoque, medias, ref, dias)
    candidatas = candidatas_iniciais(estoque, medias, diagnostico, ref, dias)
    plano, final, alternativas = construir_plano(estoque, medias, ref, dias)
    return dict(estoque=estoque, historico=historico, medias=medias, diagnostico=diagnostico,
        combinacoes=diagnosticar_combinacoes(estoque, medias, dias), candidatas=candidatas,
        plano=plano, final=final, alternativas=alternativas)


## 23. Resumo da execução
As contagens descrevem este experimento. Um lote com sugestão ainda pode manter parte do risco. Não somo quantidades de medicamentos diferentes como se fossem doses equivalentes.


In [37]:
contagens = pd.DataFrame([
    ('Unidades',len(df_cadastro_unidades)),('Produtos do recorte',len(df_medicamentos)),
    ('Lotes consolidados',len(estoque)),('Combinações unidade–medicamento',len(medias)),
    ('Lotes em risco potencial',int(diagnostico.risco.gt(TOL).sum())),
    ('Movimentos recomendados',len(plano)),('Lotes originais com recomendação',plano.id_lote_origem.nunique()),
    ('Combinações sem estoque e com consumo',int(combinacoes.situacao.eq('INDISPONIVEL_COM_CONSUMO').sum()))
],columns=['indicador','quantidade'])
display(contagens)


,indicador,quantidade
0,Unidades,17
1,Produtos do recorte,161
2,Lotes consolidados,3846
3,Combinações unidade–medicamento,2737
4,Lotes em risco potencial,1198
5,Movimentos recomendados,1076
6,Lotes originais com recomendação,797
7,Combinações sem estoque e com consumo,688


## 24. Avaliação — CRISP-DM
Verifico se as regras produzem os resultados esperados.

| Parte | Verificação |
|---|---|
| Regras | Cenários C1–C8, casos de borda e invariantes |
| Processo | Reprodução em outra sessão |
| Premissas | Sensibilidade de um parâmetro por vez |

Aceito diferença de até `0,00000001` apenas para arredondamento numérico. Categorias, IDs e movimentos devem coincidir exatamente. A contagem de cenários conformes é descritiva.

### 24.1 Cenários C1–C8
As condições seguem o Quadro 4 do rascunho V2. As referências numéricas e as contas estão registradas antes da chamada ao modelo; não são produzidas pelo gerador. Esses registros não comprovam, por si só, uma conferência humana independente.

Cada lote usa `(ID, unidade, quantidade, dias até a validade)`. O histórico tem seis saídas mensais e o medicamento fictício recebe ID 1.


In [38]:
# Mantenho os valores esperados separados da execução do modelo.
# Cada caso usa lotes fictícios e um medicamento.
CASOS = [
 {'id':'C1', 'condicao':'Ausência de risco', 'lotes':[(101,1,60,60)],
  'historico':{1:[30]*6},
  'memoria':'CMM=180/6=30; capacidade=30×60/30=60; consumível=60; risco=60−60=0.',
  'esperado':{'cmm:1':30,'consumivel:101':60,'risco:101':0,'cobertura_dias:1':60,'movimentos':''}},
 {'id':'C2', 'condicao':'Risco de vencimento', 'lotes':[(101,1,100,60)],
  'historico':{1:[20,25,30,30,35,40]},
  'memoria':'CMM=(20+25+30+30+35+40)/6=30; capacidade=60; risco=100−60=40.',
  'esperado':{'cmm:1':30,'consumivel:101':60,'risco:101':40,'candidatas:101':'','movimentos':''}},
 {'id':'C3', 'condicao':'Risco sem possibilidade de remanejamento',
  'lotes':[(101,1,100,60),(102,2,60,60)], 'historico':{1:[30]*6,2:[30]*6},
  'memoria':'Origem: risco=100−60=40. Destino: capacidade=60, estoque utilizável=60; livre=0.',
  'esperado':{'risco:101':40,'risco:102':0,'capacidade_livre:101:2':0,
              'quantidade_admissivel:101:2':0,'candidatas:101':'','movimentos':''}},
 {'id':'C4', 'condicao':'Risco com uma unidade candidata',
  'lotes':[(101,1,100,60),(102,2,20,60)], 'historico':{1:[30]*6,2:[40]*6},
  'memoria':'Origem: risco=100−60=40. Destino: 40×60/30−20=60 livres. Envio=min(40,60)=40.',
  'esperado':{'risco:101':40,'capacidade_livre:101:2':60,'capacidade_sem_novo_risco:101:2':60,
              'quantidade_admissivel:101:2':40,'candidatas:101':'2','movimentos':'101:1>2:40'}},
 {'id':'C5', 'condicao':'Múltiplas unidades candidatas',
  'lotes':[(101,1,150,60),(102,2,20,60),(103,3,10,60)],
  'historico':{1:[30]*6,2:[30]*6,3:[30]*6},
  'memoria':'Risco=150−60=90. UBS 2: 60−20=40 livres. UBS 3: 60−10=50. Escolha 3 para 50; restam 40 para 2.',
  'esperado':{'risco:101':90,'capacidade_livre:101:2':40,'capacidade_livre:101:3':50,
    'quantidade_admissivel:101:2':40,'quantidade_admissivel:101:3':50,
    'candidatas:101':'3,2','movimentos':'101:1>3:50;101:1>2:40'}},
 {'id':'C6', 'condicao':'Múltiplos lotes e PVPS',
  'lotes':[(101,1,80,30),(102,1,50,60)], 'historico':{1:[30]*6,2:[30]*6},
  'memoria':'Lote 101: capacidade 30, consumível 30, risco 50. Lote 102: capacidade acumulada 60−30 já alocados=30, risco 20. Destino recebe 30 no prazo 30 e mais 20 no prazo 60.',
  'esperado':{'consumivel:101':30,'risco:101':50,'ordem_pvps:101':1,
    'consumivel:102':30,'risco:102':20,'ordem_pvps:102':2,'movimentos':'101:1>2:30;102:1>2:20'}},
 {'id':'C7', 'condicao':'Ausência de estoque com consumo', 'lotes':[], 'historico':{1:[30]*6},
  'memoria':'Estoque=0; CMM=30; cobertura=0/30×30=0 dias. A combinação existe no histórico, mesmo sem lote.',
  'esperado':{'cmm:1':30,'cobertura_dias:1':0,'situacao:1':'INDISPONIVEL_COM_CONSUMO','movimentos':''}},
 {'id':'C8', 'condicao':'Estoque sem consumo de referência', 'lotes':[(101,1,40,60)],
  'historico':{1:[0]*6},
  'memoria':'CMM=0; capacidade=0; risco=40. Cobertura não calculável: não dividir por zero.',
  'esperado':{'cmm:1':0,'consumivel:101':0,'risco:101':40,'cobertura_dias:1':'NAO_CALCULAVEL',
              'situacao:1':'ESTOQUE_SEM_CONSUMO','movimentos':''}}
]


### 24.2 Casos de borda
B1–B5 cobrem as situações pedidas na orientação. B6–B10 verificam lote posterior, disputa de capacidade, quantidade inteira, falta de prazo útil e empate entre destinos.


In [39]:
CASOS += [
 {'id':'B1', 'condicao':'Capacidade exatamente igual à quantidade em risco',
  'lotes':[(101,1,90,60),(102,2,30,60)], 'historico':{1:[30]*6,2:[30]*6},
  'memoria':'Risco da origem=90−60=30; capacidade do destino=60−30=30. Igualdade aceita: enviar 30.',
  'esperado':{'risco:101':30,'capacidade_livre:101:2':30,'quantidade_admissivel:101:2':30,'movimentos':'101:1>2:30'}},
 {'id':'B2', 'condicao':'Empate de validade entre lotes',
  'lotes':[(102,1,40,30),(101,1,20,30)], 'historico':{1:[30]*6,2:[30]*6},
  'memoria':'Mesma validade: menor ID primeiro, apesar da entrada invertida. ID 101 usa 20; ID 102 usa 10 das 30 unidades disponíveis. Risco 30.',
  'esperado':{'ordem_pvps:101':1,'ordem_pvps:102':2,'consumivel:101':20,'consumivel:102':10,
              'risco:101':0,'risco:102':30,'movimentos':'102:1>2:30'}},
 {'id':'B3', 'condicao':'Validade muito curta: um dia', 'lotes':[(101,1,5,1)],
  'historico':{1:[30]*6,2:[60]*6},
  'memoria':'Origem utiliza 30×1/30=1, risco 4. Destino pode utilizar 60×1/30=2. Envio 2; a logística real não é modelada.',
  'esperado':{'consumivel:101':1,'risco:101':4,'capacidade_livre:101:2':2,'movimentos':'101:1>2:2'}},
 {'id':'B4', 'condicao':'Candidata com estoque superior à capacidade',
  'lotes':[(101,1,100,60),(102,2,100,60)], 'historico':{1:[30]*6,2:[30]*6},
  'memoria':'Ambas têm capacidade 60 e saldo 100. Cada uma tem risco 40, mas não tem capacidade livre para receber.',
  'esperado':{'risco:101':40,'risco:102':40,'capacidade_livre:101:2':0,'capacidade_livre:102:1':0,'movimentos':''}},
 {'id':'B5', 'condicao':'Candidata com CMM zero', 'lotes':[(101,1,100,60)],
  'historico':{1:[30]*6,2:[0]*6},
  'memoria':'Risco 40 na origem. Destino sem consumo: capacidade zero e exclusão das candidatas admissíveis.',
  'esperado':{'risco:101':40,'capacidade_livre:101:2':0,'quantidade_admissivel:101:2':0,
              'candidatas:101':'','movimentos':''}},
 {'id':'B6', 'condicao':'Recebimento deslocaria risco para lote posterior',
  'lotes':[(101,1,100,30),(102,2,60,60)], 'historico':{1:[30]*6,2:[30]*6},
  'memoria':'Destino tem 30 livres no dia 30, mas todo o consumo de 60 dias já é necessário para seu lote de 60. Folga posterior=60−60=0. Capacidade sem novo risco=0.',
  'esperado':{'risco:101':70,'capacidade_livre:101:2':30,'capacidade_sem_novo_risco:101:2':0,
              'quantidade_admissivel:101:2':0,'movimentos':''}},
 {'id':'B7', 'condicao':'Duas origens disputam a mesma capacidade',
  'lotes':[(101,1,60,30),(102,2,60,30)], 'historico':{1:[30]*6,2:[30]*6,3:[30]*6},
  'memoria':'Risco 30 em cada origem. Destino 3 comporta 30. Lote 101 é analisado primeiro e ocupa 30; lote 102 não reutiliza essa capacidade.',
  'esperado':{'risco:101':30,'risco:102':30,'quantidade_admissivel:101:3':30,
              'quantidade_admissivel:102:3':30,'movimentos':'101:1>3:30'}},
 {'id':'B8', 'condicao':'Risco fracionário e quantidade inteira', 'lotes':[(101,1,10,31)],
  'historico':{1:[7]*6,2:[3]*6},
  'memoria':'Consumível=7×31/30=217/30; risco=10−217/30=83/30. Destino comporta 3,1. Piso do mínimo=2.',
  'esperado':{'consumivel:101':217/30,'risco:101':83/30,'capacidade_livre:101:2':3.1,
              'quantidade_admissivel:101:2':2,'movimentos':'101:1>2:2'}},
 {'id':'B9', 'condicao':'Validade na data de referência', 'lotes':[(101,1,5,0)],
  'historico':{1:[30]*6,2:[60]*6},
  'memoria':'Prazo útil zero: consumível zero, risco cinco e nenhuma recomendação de envio.',
  'esperado':{'consumivel:101':0,'risco:101':5,'quantidade_admissivel:101:2':0,'movimentos':''}},
 {'id':'B10', 'condicao':'Empate entre unidades candidatas', 'lotes':[(101,1,100,60)],
  'historico':{1:[30]*6,2:[30]*6,3:[30]*6},
  'memoria':'Risco 40; destinos 2 e 3 têm capacidade 60 e quantidade admissível 40. Desempate pelo menor ID: 2.',
  'esperado':{'candidatas:101':'2,3','quantidade_admissivel:101:2':40,
              'quantidade_admissivel:101:3':40,'movimentos':'101:1>2:40'}}
]


In [40]:
referencias = pd.DataFrame([{'cenario':c['id'],'condicao':c['condicao'],
    'entradas':json.dumps({'lotes':c['lotes'],'historico':c['historico']},ensure_ascii=False),
    'memoria_calculo':c['memoria'],'esperado':json.dumps(c['esperado'],ensure_ascii=False)} for c in CASOS])
print('Referências fixadas antes da execução dos casos:')
display(referencias[['cenario','condicao','memoria_calculo']])


Referências fixadas antes da execução dos casos:


,cenario,condicao,memoria_calculo
0,C1,Ausência de risco,CMM=180/6=30; capacidade=30×60/30=60; consumív...
1,C2,Risco de vencimento,CMM=(20+25+30+30+35+40)/6=30; capacidade=60; r...
2,C3,Risco sem possibilidade de remanejamento,Origem: risco=100−60=40. Destino: capacidade=6...
3,C4,Risco com uma unidade candidata,Origem: risco=100−60=40. Destino: 40×60/30−20=...
4,C5,Múltiplas unidades candidatas,Risco=150−60=90. UBS 2: 60−20=40 livres. UBS 3...
5,C6,Múltiplos lotes e PVPS,"Lote 101: capacidade 30, consumível 30, risco ..."
6,C7,Ausência de estoque com consumo,Estoque=0; CMM=30; cobertura=0/30×30=0 dias. A...
7,C8,Estoque sem consumo de referência,CMM=0; capacidade=0; risco=40. Cobertura não c...
8,B1,Capacidade exatamente igual à quantidade em risco,Risco da origem=90−60=30; capacidade do destin...
9,B2,Empate de validade entre lotes,"Mesma validade: menor ID primeiro, apesar da e..."


#### Funções de verificação
Aplico as mesmas condições de consistência aos cenários e à base completa. Os valores esperados são lidos diretamente das referências.


In [41]:
def verificar_invariantes(estoque, medias, diagnostico, candidatas, plano, final, referencia, dias_mes):
    """Confiro as condições em todos os registros."""
    registros = []
    def registrar(regra, numero, condicao):
        registros.append({'regra': regra, 'registros_verificados': int(numero),
                          'conforme': bool(condicao)})
    q, r = diagnostico.quantidade, diagnostico.risco
    registrar('Risco entre zero e saldo do lote', len(diagnostico),
              ((r >= -TOL) & (r <= q + TOL)).all())
    registrar('Consumível + risco = saldo', len(diagnostico),
              np.allclose(diagnostico.consumivel + r, q, atol=TOL, rtol=0))
    ordem_ok, capacidade_ok, grupos_verificados = True, True, 0
    for _, g in diagnostico.groupby(CHAVE):
        grupos_verificados += 1
        ordenado = g.sort_values(['validade', 'id_lote'])
        ordem_ok &= ordenado.ordem_pvps.tolist() == list(range(1, len(g)+1))
        acumulado = ordenado.consumivel.cumsum()
        capacidade_ok &= (acumulado <= ordenado.capacidade_acumulada + TOL).all()
        capacidade_ok &= np.allclose(acumulado - ordenado.consumivel,
                                    ordenado.consumo_anterior, atol=TOL, rtol=0)
    registrar('Ordem PVPS por validade e ID', grupos_verificados, ordem_ok)
    registrar('Capacidade não duplicada entre lotes do grupo', grupos_verificados, capacidade_ok)
    for campo in ['capacidade_livre', 'capacidade_sem_novo_risco', 'quantidade_admissivel']:
        registrar(campo + ' não negativa', len(candidatas), candidatas[campo].ge(-TOL).all())
    registrar('Origem diferente de toda candidata', len(candidatas),
              candidatas.id_unidade_origem.ne(candidatas.id_unidade_destino).all())
    antes = estoque.groupby('id_medicamento').quantidade.sum().sort_index()
    depois = final.groupby('id_medicamento').quantidade.sum().reindex(antes.index, fill_value=0)
    registrar('Conservação por medicamento após plano', len(antes),
              np.allclose(antes.astype(float), depois.astype(float), atol=TOL, rtol=0))
    registrar('Saldo final não negativo', len(final), final.quantidade.ge(-TOL).all())
    registrar('Movimentos inteiros positivos', len(plano),
              (plano.quantidade_admissivel.ge(1) & plano.quantidade_admissivel.mod(1).eq(0)).all())
    registrar('Quantidade enviada não supera o risco disponível', len(plano),
              (plano.quantidade_admissivel <= plano.risco_antes + TOL).all())
    registrar('Consumo da origem preservado em cada movimento', len(plano),
              np.allclose(plano.consumo_origem_antes.astype(float), plano.consumo_origem_depois.astype(float), atol=TOL, rtol=0))
    registrar('Risco do destino não aumenta em cada movimento', len(plano),
              (plano.risco_destino_depois <= plano.risco_destino_antes + TOL).all())
    registrar('Cada lote recebido possui um único ID', len(plano), plano.id_lote_recebido.is_unique)
    # Reconstruo os saldos a partir dos movimentos registrados.
    reconstruido = estoque.set_index('id_lote').quantidade.to_dict()
    for p in plano.itertuples(index=False):
        reconstruido[p.id_lote_origem] -= p.quantidade_admissivel
        reconstruido[p.id_lote_recebido] = p.quantidade_admissivel
    observado = final.set_index('id_lote').quantidade
    registrar('Saldo final corresponde aos débitos e créditos', len(final),
              set(reconstruido) == set(observado.index) and
              np.allclose([reconstruido[i] for i in observado.index], observado.astype(float), atol=TOL, rtol=0))
    # Confiro também a capacidade usada pelos recebimentos.
    diag_final = diagnosticar_lotes(final, medias, referencia, dias_mes)
    cumulativo_ok = all((g.consumivel.cumsum() <= g.capacidade_acumulada + TOL).all()
                       for _, g in diag_final.groupby(CHAVE))
    registrar('Capacidade final não duplicada após todos os recebimentos', len(diag_final), cumulativo_ok)
    return pd.DataFrame(registros)


In [42]:
def executar_referencias(casos, referencia):
    comparacoes, resumos = [], []
    periodos = pd.period_range(end=pd.Timestamp(referencia).to_period('M')-1, periods=6)
    for caso in casos:
        estoque = pd.DataFrame(caso['lotes'], columns=['id_lote', 'id_unidade', 'quantidade', 'dias'])
        estoque['id_medicamento'] = 1
        estoque['validade'] = referencia + pd.to_timedelta(estoque.pop('dias'), unit='D')
        estoque = estoque[CAMPOS_ESTOQUE]
        historico = pd.DataFrame([
            (u, 1, str(m), q) for u, valores in caso['historico'].items()
            for m, q in zip(periodos, valores)], columns=CHAVE+['periodo', 'quantidade_saida'])
        medias = calcular_cmm(historico, periodos)
        diag = diagnosticar_lotes(estoque, medias, referencia)
        comb = diagnosticar_combinacoes(estoque, medias)
        candidatas = candidatas_iniciais(estoque, medias, diag, referencia)
        plano, final, _ = construir_plano(estoque, medias, referencia)
        obtidos = {'movimentos': ';'.join(
            f'{int(p.id_lote_origem)}:{int(p.id_unidade_origem)}>{int(p.id_unidade_destino)}:{int(p.quantidade_admissivel)}'
            for p in plano.itertuples(index=False))}
        for d in diag.itertuples(index=False):
            for campo in ['consumivel', 'risco', 'ordem_pvps']:
                obtidos[f'{campo}:{d.id_lote}'] = getattr(d, campo)
        for d in comb.itertuples(index=False):
            for campo in ['cmm', 'cobertura_dias', 'situacao']:
                valor = getattr(d, campo)
                obtidos[f'{campo}:{d.id_unidade}'] = 'NAO_CALCULAVEL' if pd.isna(valor) else valor
        for d in candidatas.itertuples(index=False):
            for campo in ['capacidade_livre', 'capacidade_sem_novo_risco', 'quantidade_admissivel']:
                obtidos[f'{campo}:{d.id_lote_origem}:{d.id_unidade_destino}'] = getattr(d, campo)
        for ident, g in candidatas.groupby('id_lote_origem', sort=False):
            obtidos[f'candidatas:{ident}'] = ','.join(str(int(v)) for v in
                g.loc[g.quantidade_admissivel.gt(0), 'id_unidade_destino'])
        for ident in diag.id_lote:
            obtidos.setdefault(f'candidatas:{ident}', '')
        for campo, esperado in caso['esperado'].items():
            obtido = obtidos.get(campo, 'CAMPO_AUSENTE')
            numerico = isinstance(esperado, (int, float))
            conforme = (isinstance(obtido, (int, float, np.number)) and
                        np.isclose(obtido, esperado, atol=TOL, rtol=0)) if numerico else obtido == esperado
            comparacoes.append({'cenario': caso['id'], 'campo': campo,
                'esperado': esperado, 'obtido': obtido, 'conforme': bool(conforme)})
        invariantes = verificar_invariantes(estoque, medias, diag, candidatas, plano, final, referencia, 30)
        itens = [r for r in comparacoes if r['cenario'] == caso['id']]
        resumos.append({'cenario': caso['id'], 'condicao': caso['condicao'],
            'campos_comparados': len(itens), 'divergencias': sum(not r['conforme'] for r in itens),
            'invariantes_conformes': bool(invariantes.conforme.all()),
            'conforme': all(r['conforme'] for r in itens) and bool(invariantes.conforme.all())})
    return pd.DataFrame(comparacoes), pd.DataFrame(resumos)


In [43]:
comparacao_cenarios, resumo_cenarios = executar_referencias(CASOS, DATA_REFERENCIA)
obtidos_por_cenario = pd.Series({ident:json.dumps(dict(zip(g.campo,g.obtido)),ensure_ascii=False)
    for ident,g in comparacao_cenarios.groupby('cenario')},name='obtido').rename_axis('cenario')
quadro_cenarios = referencias.merge(obtidos_por_cenario,on='cenario').merge(
    resumo_cenarios[['cenario','conforme']],on='cenario')
display(resumo_cenarios)
print('Cenários C conformes:',int(resumo_cenarios.query("cenario.str.startswith('C')",engine='python').conforme.sum()),'de 8')
print('Bordas conformes:',int(resumo_cenarios.query("cenario.str.startswith('B')",engine='python').conforme.sum()),'de 10')
exigir(resumo_cenarios.conforme.all(), 'Há cenário divergente; revise antes de usar os resultados.')


,cenario,condicao,campos_comparados,divergencias,invariantes_conformes,conforme
0,C1,Ausência de risco,5,0,True,True
1,C2,Risco de vencimento,5,0,True,True
2,C3,Risco sem possibilidade de remanejamento,6,0,True,True
3,C4,Risco com uma unidade candidata,6,0,True,True
4,C5,Múltiplas unidades candidatas,7,0,True,True
5,C6,Múltiplos lotes e PVPS,7,0,True,True
6,C7,Ausência de estoque com consumo,4,0,True,True
7,C8,Estoque sem consumo de referência,6,0,True,True
8,B1,Capacidade exatamente igual à quantidade em risco,4,0,True,True
9,B2,Empate de validade entre lotes,7,0,True,True


Cenários C conformes: 8 de 8
Bordas conformes: 10 de 10


### 24.3 Invariantes da base completa
Confiro condições que devem valer para todos os registros. A conservação é verificada por medicamento, inclusive após o plano.


In [44]:
invariantes = verificar_invariantes(estoque, medias, diagnostico, candidatas, plano,
    estoque_final, DATA_REFERENCIA, PARAMETROS['dias_mes'])
display(invariantes)
exigir(invariantes.conforme.all(), 'Há invariante divergente na base completa.')


,regra,registros_verificados,conforme
0,Risco entre zero e saldo do lote,3846,True
1,Consumível + risco = saldo,3846,True
2,Ordem PVPS por validade e ID,1916,True
3,Capacidade não duplicada entre lotes do grupo,1916,True
4,capacidade_livre não negativa,19168,True
5,capacidade_sem_novo_risco não negativa,19168,True
6,quantidade_admissivel não negativa,19168,True
7,Origem diferente de toda candidata,19168,True
8,Conservação por medicamento após plano,161,True
9,Saldo final não negativo,4922,True


### 24.4 Consulta dos cenários no Colab
Em `CENARIO_EXIBIDO`, escolho C1–C8 ou B1–B10. A célula mostra entradas, contas, esperado, obtido e conformidade.

A conferência independente por planilha do item 3.4 não integra esta versão. Esta consulta não substitui essa etapa; a retirada ainda precisa ser alinhada com o orientador.


In [45]:
CENARIO_EXIBIDO = 'C4'  # C1–C8 ou B1–B10
caso = next((c for c in CASOS if c['id'] == CENARIO_EXIBIDO), None)
exigir(caso is not None, 'Cenário inexistente. Use C1–C8 ou B1–B10.')
print(caso['id'], '—', caso['condicao'])
print('Entradas de estoque (todos os lotes pertencem ao medicamento fictício 1):')
display(pd.DataFrame(caso['lotes'], columns=['id_lote','id_unidade','quantidade','dias_ate_validade']))
print('Seis saídas mensais de cada unidade:')
display(pd.DataFrame.from_dict(caso['historico'], orient='index',
    columns=[str(p) for p in periodos]).rename_axis('id_unidade'))
print('Memória de cálculo:', caso['memoria'])
print('Comparação por campo:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None,
                       'display.max_columns', None, 'display.float_format', lambda x: f'{x:.8f}'):
    display(comparacao_cenarios.loc[comparacao_cenarios.cenario.eq(CENARIO_EXIBIDO),
        ['campo','esperado','obtido','conforme']])


C4 — Risco com uma unidade candidata
Entradas de estoque (todos os lotes pertencem ao medicamento fictício 1):


,id_lote,id_unidade,quantidade,dias_ate_validade
0,101,1,100,60
1,102,2,20,60


Seis saídas mensais de cada unidade:


,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05
id_unidade,,,,,,
1,30,30,30,30,30,30
2,40,40,40,40,40,40


Memória de cálculo: Origem: risco=100−60=40. Destino: 40×60/30−20=60 livres. Envio=min(40,60)=40.
Comparação por campo:


,campo,esperado,obtido,conforme
16,risco:101,40,40.00000000,True
17,capacidade_livre:101:2,60,60.00000000,True
18,capacidade_sem_novo_risco:101:2,60,60.00000000,True
19,quantidade_admissivel:101:2,40,40,True
20,candidatas:101,2,2,True
21,movimentos,101:1>2:40,101:1>2:40,True


### 24.5 Sensibilidade das premissas
Comparo a base com oito variações, mudando um fator por vez: sal do hash, faixa de validade, percentual de consumo-base zero e dias por mês.

Uso validades de 15–270 e 60–1080 dias; consumo-base zero de 0% e 30%; meses de 28 e 31 dias. Mudar o sal altera validade e histórico. São variações experimentais, não faixas recomendadas para UBS.

Comparo lotes em risco, presença de candidatas e mudanças nos conjuntos. As candidatas são sempre calculadas no estado inicial. A permanência de uma conclusão geral não garante que cada recomendação continue igual.


In [46]:
def analisar_sensibilidade(base, grade_original, parametros, principal):
    # Mudo um fator por vez.
    variantes = [
        ('BASE', 'nenhum', {}),
        ('SAL_A', 'sal', {'sal':parametros['sal']+'_A'}),
        ('SAL_B', 'sal', {'sal':parametros['sal']+'_B'}),
        ('VALIDADE_CURTA', 'faixa de validade', {'validade_min':15, 'validade_max':270}),
        ('VALIDADE_LONGA', 'faixa de validade', {'validade_min':60, 'validade_max':1080}),
        ('ZERO_0', 'percentual de consumo-base zero', {'percentual_zero':0}),
        ('ZERO_30', 'percentual de consumo-base zero', {'percentual_zero':30}),
        ('MES_28', 'dias por mês', {'dias_mes':28.0}),
        ('MES_31', 'dias por mês', {'dias_mes':31.0})]
    def conjuntos(execucao):
        risco = set(execucao['diagnostico'].query('risco > @TOL').id_lote)
        c = execucao['candidatas'].query('quantidade_admissivel > 0')
        pares = set(map(tuple, c[['id_lote_origem','id_unidade_destino']].to_numpy()))
        return risco, pares
    risco_base, pares_base = conjuntos(principal)
    linhas = []
    for nome, fator, mudancas in variantes:
        p = dict(parametros, **mudancas)
        resultado = principal if nome == 'BASE' else executar_cenario_base(base, grade_original, p)
        risco, pares = conjuntos(resultado)
        lotes_com_candidata = {lote for lote, destino in pares}
        regras = verificar_invariantes(resultado['estoque'], resultado['medias'], resultado['diagnostico'],
            resultado['candidatas'], resultado['plano'], resultado['final'], pd.Timestamp(p['referencia']), p['dias_mes'])
        exigir(regras.conforme.all(), f'Invariante falhou na sensibilidade: {nome}')
        linhas.append({'cenario':nome, 'fator_alterado':fator,
            'valor':json.dumps(mudancas, ensure_ascii=False), 'lotes_em_risco':len(risco),
            'lotes_com_candidata':len(lotes_com_candidata), 'lotes_sem_candidata':len(risco-lotes_com_candidata),
            'pares_lote_destino':len(pares), 'movimentos':len(resultado['plano']),
            'risco_entraram':len(risco-risco_base), 'risco_sairam':len(risco_base-risco),
            'candidatas_entraram':len(pares-pares_base), 'candidatas_sairam':len(pares_base-pares),
            'consumo_zero_observado':int(resultado['medias'].cmm.eq(0).sum()),
            'regras_conformes':True})
        print('Sensibilidade concluída:', nome)
    return pd.DataFrame(linhas)


In [47]:
sensibilidade = analisar_sensibilidade(df_estoque_consolidado, grade, PARAMETROS, resultado)
display(sensibilidade.drop(columns=['valor']))
qualitativo = pd.DataFrame({
    'conclusao':['Existe risco potencial','Existe lote em risco com candidata','Existe risco sem candidata'],
    'configuracoes_em_que_ocorre':[
        int(sensibilidade.lotes_em_risco.gt(0).sum()),
        int(sensibilidade.lotes_com_candidata.gt(0).sum()),
        int(sensibilidade.lotes_sem_candidata.gt(0).sum())],
    'total_configuracoes':len(sensibilidade)})
display(qualitativo)
print('Identidades e contagens mudam com as premissas. Persistência de uma conclusão geral não garante estabilidade de cada recomendação.')


Sensibilidade concluída: BASE
Sensibilidade concluída: SAL_A
Sensibilidade concluída: SAL_B
Sensibilidade concluída: VALIDADE_CURTA
Sensibilidade concluída: VALIDADE_LONGA
Sensibilidade concluída: ZERO_0
Sensibilidade concluída: ZERO_30
Sensibilidade concluída: MES_28
Sensibilidade concluída: MES_31


,cenario,fator_alterado,lotes_em_risco,lotes_com_candidata,lotes_sem_candidata,pares_lote_destino,movimentos,risco_entraram,risco_sairam,candidatas_entraram,candidatas_sairam,consumo_zero_observado,regras_conformes
0,BASE,nenhum,1198,1166,32,10234,1076,0,0,0,0,417,True
1,SAL_A,sal,1230,1127,103,10500,1130,620,588,8227,7961,437,True
2,SAL_B,sal,1228,1170,58,11048,1239,629,599,8675,7861,415,True
3,VALIDADE_CURTA,faixa de validade,1537,1325,212,11593,1455,394,55,3842,2483,417,True
4,VALIDADE_LONGA,faixa de validade,954,952,2,9937,984,47,291,1948,2245,417,True
5,ZERO_0,percentual de consumo-base zero,790,756,34,6045,715,0,408,841,5030,23,True
6,ZERO_30,percentual de consumo-base zero,1730,1698,32,13621,1471,532,0,5335,1948,860,True
7,MES_28,dias por mês,1171,1140,31,10185,1075,0,27,156,205,417,True
8,MES_31,dias por mês,1206,1177,29,10246,1082,8,0,101,89,417,True


,conclusao,configuracoes_em_que_ocorre,total_configuracoes
0,Existe risco potencial,9,9
1,Existe lote em risco com candidata,9,9
2,Existe risco sem candidata,9,9


Identidades e contagens mudam com as premissas. Persistência de uma conclusão geral não garante estabilidade de cada recomendação.


### 24.6 Reprodutibilidade
Comparo arquivos, parâmetros, versões das bibliotecas e o conteúdo completo de nove tabelas.

1. Executo tudo para criar `referencia_reprodutibilidade.json`.
2. Reinicio a sessão do Colab, mantendo os arquivos, e executo novamente.
3. Confiro `SESSAO_DISTINTA` e `True` nas nove tabelas.

`MESMA_SESSAO` indica que ainda falta reiniciar. `CONTEXTO_DIFERENTE` indica mudança nos arquivos, parâmetros ou ambiente. A referência anterior é preservada; só inicio outra quando a mudança é intencional.


In [48]:
def assinatura_tabela(tabela, chaves):
    """Comparo o conteúdo completo, sem arredondar os valores."""
    canonica = tabela.sort_values(chaves).reset_index(drop=True)
    texto = canonica.to_csv(index=False, lineterminator='\n', date_format='%Y-%m-%d', float_format='%.17g')
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()


In [49]:
def registrar_reprodutibilidade(resultado, parametros, arquivos, caminho, id_sessao):
    tabelas = {'estoque':['id_lote'], 'historico':CHAVE+['periodo'],
        'medias':CHAVE, 'diagnostico':['id_lote'], 'combinacoes':CHAVE,
        'candidatas':['id_lote_origem','id_unidade_destino'], 'plano':['ordem_movimento'],
        'final':['id_lote'], 'alternativas':['etapa','id_unidade_destino']}
    contexto = {'parametros':parametros,
        'entradas':{Path(p).name:hashlib.sha256(Path(p).read_bytes()).hexdigest() for p in arquivos},
        'python':sys.version.split()[0], 'pandas':pd.__version__, 'numpy':np.__version__}
    hashes = {nome:assinatura_tabela(resultado[nome], chaves) for nome,chaves in tabelas.items()}
    atual = dict(contexto=contexto, hashes=hashes, sessao=id_sessao)
    anterior = json.loads(caminho.read_text()) if caminho.exists() else None
    if anterior is None:
        caminho.write_text(json.dumps(atual, indent=2, ensure_ascii=False))
        status = 'PRIMEIRA_EXECUCAO'
        print('Referência gravada. Reinicie a sessão e execute tudo novamente, mantendo os arquivos.')
    elif anterior['contexto'] != contexto:
        status = 'CONTEXTO_DIFERENTE'
        print('Entradas, parâmetros ou bibliotecas mudaram. Referência preservada; não é uma repetição comparável.')
    elif anterior['sessao'] == id_sessao:
        status = 'MESMA_SESSAO'
        print('Reexecução na mesma sessão. Ainda é necessário reiniciar o ambiente.')
    else:
        status = 'SESSAO_DISTINTA'
    linhas = []
    for nome in hashes:
        comparavel = anterior is not None and anterior['contexto'] == contexto
        igual = hashes[nome] == anterior['hashes'].get(nome) if comparavel else None
        linhas.append({'tabela':nome, 'situacao':status, 'resultado_identico':igual,
                       'sha256':hashes[nome]})
    tabela = pd.DataFrame(linhas)
    if status == 'SESSAO_DISTINTA':
        exigir(tabela.resultado_identico.all(), 'Resultado mudou após reinício. Investigue antes de usar.')
        print('Conteúdo completo das nove tabelas idêntico em sessões distintas.')
    return tabela


In [50]:
reprodutibilidade = registrar_reprodutibilidade(resultado, PARAMETROS,
    [PASTA_DADOS/nome for nome in ARQUIVOS], Path('referencia_reprodutibilidade.json'), _ID_SESSAO)
display(reprodutibilidade[['tabela','situacao','resultado_identico']])


Conteúdo completo das nove tabelas idêntico em sessões distintas.


,tabela,situacao,resultado_identico
0,estoque,SESSAO_DISTINTA,True
1,historico,SESSAO_DISTINTA,True
2,medias,SESSAO_DISTINTA,True
3,diagnostico,SESSAO_DISTINTA,True
4,combinacoes,SESSAO_DISTINTA,True
5,candidatas,SESSAO_DISTINTA,True
6,plano,SESSAO_DISTINTA,True
7,final,SESSAO_DISTINTA,True
8,alternativas,SESSAO_DISTINTA,True


### 24.7 Limites da avaliação
Os testes verificam a implementação das regras dentro das premissas adotadas.

- Validade e consumo são sintéticos; não representam dados observados nas UBS.
- O CMM permanece constante e não há reposição, transporte ou armazenamento no cálculo.
- A grade inclui todas as combinações, sem verificar a elegibilidade operacional de cada unidade para cada produto.
- A ordem das escolhas afeta o plano; não há busca da melhor solução global.
- Os resultados não comprovam redução real de perdas nem determinam que uma transferência deva ocorrer.

A pendência da conferência independente está registrada na seção 24.4.


## 25. Consulta e exportação
### 25.1 Consulta do plano
Os filtros consultam o plano pronto. Uso `None` para mostrar todas as opções. As quantidades se referem à posição do experimento.


In [51]:
UNIDADE_ORIGEM = None
UNIDADE_DESTINO = None
MEDICAMENTO = None
LOTE_ORIGEM = None
nomes = df_cadastro_unidades.set_index('id_unidade').nome_unidade
catalogo = df_medicamentos.set_index('id_medicamento').descricao_produto
plano_apresentacao = plano.copy()
plano_apresentacao['origem'] = plano_apresentacao.id_unidade_origem.map(nomes)
plano_apresentacao['destino'] = plano_apresentacao.id_unidade_destino.map(nomes)
plano_apresentacao['medicamento'] = plano_apresentacao.id_medicamento.map(catalogo)
consulta = plano_apresentacao.copy()
for campo,valor in [('id_unidade_origem',UNIDADE_ORIGEM),('id_unidade_destino',UNIDADE_DESTINO),
                    ('id_medicamento',MEDICAMENTO),('id_lote_origem',LOTE_ORIGEM)]:
    if valor is not None:
        consulta = consulta.loc[consulta[campo].eq(valor)]
print('Movimentos no filtro:',len(consulta))
display(consulta[['ordem_movimento','id_lote_origem','origem','destino','medicamento','validade',
                  'quantidade_admissivel']].head(20))


Movimentos no filtro: 1076


,ordem_movimento,id_lote_origem,origem,destino,medicamento,validade,quantidade_admissivel
0,1,36,EAP ITOUPAVA,ESF SILVIO SCHUTZ BAIRRO SUMARE,ACIDO FOLICO 5mg (COMPRIMIDO),2026-07-10,10
1,2,3231,PSF SANTA RITA,PSF FUNDO CANOAS,ACIDO FOLICO 5mg (COMPRIMIDO),2026-07-10,6
2,3,2770,PSF PAMPLONA,PSF CANOAS,ENALAPRIL 10MG (COMPRIMIDO),2026-07-11,16
3,4,1493,PSF BUDAG,PSF SANTANA,HIDROCLOROTIAZIDA 25 MG,2026-07-12,20
4,5,149,EAP ITOUPAVA,PSF PROGRESSO,DILTIAZEM 60 MG,2026-07-12,33
5,6,149,EAP ITOUPAVA,PSF BARRAGEM CAIC,DILTIAZEM 60 MG,2026-07-12,11
6,7,3438,PSF SANTANA,PSF LARANJEIRAS,FINASTERIDA 5MG (COMPRIMIDO),2026-07-14,60
7,8,2181,PSF FUNDO CANOAS,PSF TABOAO,SINVASTATINA 20 MG,2026-07-14,5
8,9,1033,PSF BELA ALIANCA,PSF TABOAO,AMOXICILINA 500 MG + CLAVULANATO 125 MG (CÁPSU...,2026-07-16,21
9,10,3199,PSF PROGRESSO,PSF FUNDO CANOAS,AMOXICILINA 500 MG + CLAVULANATO 125 MG (CÁPSU...,2026-07-16,13


### 25.2 Detalhe de uma sugestão
Altero `MOVIMENTO` para consultar os destinos comparados no momento daquela escolha.


In [52]:
MOVIMENTO = 1
if len(plano):
    movimento = plano.loc[plano.ordem_movimento.eq(MOVIMENTO)]
    exigir(len(movimento)==1, 'Movimento inexistente.')
    m = movimento.iloc[0]
    print('Lote:',int(m.id_lote_origem),'| Destino escolhido:',int(m.id_unidade_destino),
          '| Quantidade:',int(m.quantidade_admissivel))
    display(alternativas.loc[alternativas.etapa.eq(m.etapa),
        ['id_unidade_destino','cmm_destino','capacidade_livre','capacidade_sem_novo_risco',
         'quantidade_admissivel','motivo']])
else:
    print('Nenhum movimento admissível nesta configuração.')


Lote: 36 | Destino escolhido: 2 | Quantidade: 10


,id_unidade_destino,cmm_destino,capacidade_livre,capacidade_sem_novo_risco,quantidade_admissivel,motivo
0,2,86.5000,89.3833,89.3833,10,ADMISSIVEL
1,11,85.8333,88.6944,88.6944,10,ADMISSIVEL
2,14,55.6667,57.5222,11.3222,10,ADMISSIVEL
3,8,42.6667,44.0889,29.4667,10,ADMISSIVEL
4,3,73.8333,42.2944,42.2944,10,ADMISSIVEL
5,9,29.6667,30.6556,30.6556,10,ADMISSIVEL
6,4,106.3333,109.8778,0.0000,0,CAPACIDADE_OU_RISCO_INFERIOR_A_UMA_UNIDADE
7,6,48.6667,50.2889,0.0000,0,CAPACIDADE_OU_RISCO_INFERIOR_A_UMA_UNIDADE
8,10,26.0000,26.8667,0.0000,0,CAPACIDADE_OU_RISCO_INFERIOR_A_UMA_UNIDADE
9,17,24.5000,25.3167,0.0000,0,CAPACIDADE_OU_RISCO_INFERIOR_A_UMA_UNIDADE


### 25.3 Exportação
O ZIP reúne os resultados e as verificações executadas. Os IDs permitem relacionar as tabelas.

No Colab, uso `BAIXAR_RESULTADOS = True` para baixar o arquivo.


In [53]:
PASTA_RESULTADOS = Path('resultados_modelo_final')
PASTA_RESULTADOS.mkdir(exist_ok=True)
tabelas = {
    'estoque_preparado':estoque, 'historico_sintetico':historico,
    'diagnostico_lotes':diagnostico,'diagnostico_combinacoes':combinacoes,
    'plano_remanejamento':plano_apresentacao,'alternativas_decisoes':alternativas,
    'cenarios_quadro':quadro_cenarios,'cenarios_comparacao':comparacao_cenarios,
    'cenarios_resumo':resumo_cenarios,'invariantes':invariantes,
    'sensibilidade':sensibilidade,'reprodutibilidade':reprodutibilidade}
for nome,tabela in tabelas.items():
    tabela.to_csv(PASTA_RESULTADOS/f'{nome}.csv',index=False,encoding='utf-8-sig')
(PASTA_RESULTADOS/'parametros.json').write_text(json.dumps(PARAMETROS,ensure_ascii=False,indent=2))
arquivo_zip=Path('Resultados_Modelo_Final.zip')
with zipfile.ZipFile(arquivo_zip,'w',zipfile.ZIP_DEFLATED) as z:
    arquivos_exportados = [PASTA_RESULTADOS/f'{nome}.csv' for nome in tabelas]
    arquivos_exportados.append(PASTA_RESULTADOS/'parametros.json')
    for arquivo in arquivos_exportados:
        z.write(arquivo,arquivo.relative_to(PASTA_RESULTADOS))
    z.write('referencia_reprodutibilidade.json')
print('Resultados preparados:',arquivo_zip.name)
BAIXAR_RESULTADOS = False
if BAIXAR_RESULTADOS:
    from google.colab import files
    files.download(str(arquivo_zip))


Resultados preparados: Resultados_Modelo_Final.zip
